# Chapter 3: The Density Matrix - Code Examples

The density matrix is **the** central object in quantum mechanics.

Key insight: **Diagonal = classical, Off-diagonal = quantum coherence**

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

## Helper Functions

In [ ]:
def make_pure_state_dm(psi):
    """Density matrix from state vector: ρ = |ψ⟩⟨ψ|"""
    psi = np.array(psi, dtype=complex)
    psi = psi / np.linalg.norm(psi)  # normalize
    return np.outer(psi, np.conj(psi))


def purity(rho):
    """Tr(ρ²) - equals 1 for pure states, 1/d for maximally mixed."""
    return np.real(np.trace(rho @ rho))


def von_neumann_entropy(rho):
    """S(ρ) = -Tr(ρ log₂ ρ) in bits."""
    eigenvalues = np.linalg.eigvalsh(rho)
    eigenvalues = eigenvalues[eigenvalues > 1e-10]
    return -np.sum(eigenvalues * np.log2(eigenvalues))


def is_pure(rho, tol=1e-6):
    """Check if ρ² = ρ (pure state test)."""
    return np.allclose(rho @ rho, rho, atol=tol)

## Pure State: Superposition

$$|\psi\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$$

In [ ]:
# Pure superposition
psi_super = [1, 1]  # will be normalized
rho_super = make_pure_state_dm(psi_super)

print("=== Superposition (|0⟩ + |1⟩)/√2 ===")
print(f"Density matrix:\n{rho_super}")
print()
print(f"Diagonal (probabilities): {np.diag(rho_super).real}")
print(f"Off-diagonal (coherence): {rho_super[0, 1]}")
print()
print(f"Is pure? {is_pure(rho_super)}")
print(f"Purity Tr(ρ²): {purity(rho_super):.3f}")
print(f"Eigenvalues: {np.linalg.eigvalsh(rho_super)}")
print(f"Entropy: {von_neumann_entropy(rho_super):.3f} bits")

## Mixed State: Classical Coin Flip

50% probability of $|0\rangle$, 50% probability of $|1\rangle$ — but we don't know which.

In [ ]:
# Classical mixture
rho_0 = make_pure_state_dm([1, 0])  # |0⟩
rho_1 = make_pure_state_dm([0, 1])  # |1⟩
rho_mix = 0.5 * rho_0 + 0.5 * rho_1

print("=== Classical 50/50 mixture ===")
print(f"Density matrix:\n{rho_mix}")
print()
print(f"Diagonal (probabilities): {np.diag(rho_mix).real}")
print(f"Off-diagonal (coherence): {rho_mix[0, 1]}")
print()
print(f"Is pure? {is_pure(rho_mix)}")
print(f"Purity Tr(ρ²): {purity(rho_mix):.3f}")
print(f"Eigenvalues: {np.linalg.eigvalsh(rho_mix)}")
print(f"Entropy: {von_neumann_entropy(rho_mix):.3f} bits")

## Side-by-Side Comparison

Same diagonals, completely different states!

In [ ]:
print("COMPARISON: Same diagonals, different off-diagonals")
print("=" * 55)
print()
print(f"{'Property':<20} {'Superposition':>15} {'Classical Mix':>15}")
print("-" * 55)
print(f"{'P(measure |0⟩)':<20} {rho_super[0, 0].real:>15.2f} {rho_mix[0, 0].real:>15.2f}")
print(f"{'P(measure |1⟩)':<20} {rho_super[1, 1].real:>15.2f} {rho_mix[1, 1].real:>15.2f}")
print(f"{'Off-diagonal':<20} {rho_super[0, 1].real:>15.2f} {rho_mix[0, 1].real:>15.2f}")
print(f"{'Purity':<20} {purity(rho_super):>15.2f} {purity(rho_mix):>15.2f}")
print(
    f"{'Entropy (bits)':<20} {von_neumann_entropy(rho_super):>15.2f} {von_neumann_entropy(rho_mix):>15.2f}"
)
print(f"{'Is pure?':<20} {str(is_pure(rho_super)):>15} {str(is_pure(rho_mix)):>15}")
print()
print("Key insight: Off-diagonals encode coherence. Zero off-diagonals = classical.")

## The Three Properties of Valid Density Matrices

In [ ]:
def check_valid_density_matrix(rho, name="ρ"):
    """Verify the three properties of a valid density matrix."""
    print(f"Checking {name}:")

    # 1. Hermitian: ρ† = ρ
    is_hermitian = np.allclose(rho, rho.conj().T)
    print(f"  1. Hermitian (ρ† = ρ): {is_hermitian}")

    # 2. Unit trace: Tr(ρ) = 1
    trace = np.trace(rho).real
    print(f"  2. Unit trace (Tr(ρ) = 1): {np.isclose(trace, 1)} (Tr = {trace:.6f})")

    # 3. Positive semi-definite: all eigenvalues ≥ 0
    eigenvalues = np.linalg.eigvalsh(rho)
    is_psd = np.all(eigenvalues >= -1e-10)
    print(f"  3. Positive semi-definite: {is_psd} (eigenvalues: {eigenvalues})")

    return is_hermitian and np.isclose(trace, 1) and is_psd


print("=" * 50)
check_valid_density_matrix(rho_super, "Superposition")
print()
check_valid_density_matrix(rho_mix, "Classical mixture")

## Connection to Computational Mechanics

In quantum computational mechanics, we construct:

$$\rho = \sum_j \pi_j |s_j\rangle\langle s_j|$$

where $\pi_j$ are causal state probabilities and $|s_j\rangle$ are quantum signal states.

If signal states overlap ($\langle s_i | s_j \rangle \neq 0$), we get $C_q < C_\mu$.

In [ ]:
# Example: Two causal states with non-orthogonal signal states
# Perturbed coin example: p = 0.3
p = 0.3

# Signal states (from Gu et al. 2012)
s0 = np.array([np.sqrt(1 - p), np.sqrt(p)])
s1 = np.array([np.sqrt(p), np.sqrt(1 - p)])

# Equal stationary probabilities
pi_0, pi_1 = 0.5, 0.5

# Quantum density matrix
rho_q = pi_0 * np.outer(s0, s0) + pi_1 * np.outer(s1, s1)

# Classical "density matrix" (orthogonal states)
rho_c = np.diag([pi_0, pi_1])

print("=== Perturbed Coin (p=0.3) ===")
print(f"\nSignal state |s₀⟩ = {s0}")
print(f"Signal state |s₁⟩ = {s1}")
print(f"Overlap ⟨s₀|s₁⟩ = {np.dot(s0, s1):.3f}")
print()
print(f"Quantum ρ:\n{rho_q}")
print(f"\nClassical P (diagonal):\n{rho_c}")
print()
print(f"Quantum complexity C_q = {von_neumann_entropy(rho_q):.3f} bits")
print(f"Classical complexity C_μ = {von_neumann_entropy(rho_c):.3f} bits")
print(f"Quantum advantage = {von_neumann_entropy(rho_c) - von_neumann_entropy(rho_q):.3f} bits")

## Key Takeaway

> **The density matrix captures everything about a quantum state.**
>
> - **Diagonal** = classical probabilities
> - **Off-diagonal** = quantum coherences
> - **Eigenvalues** → entropy and purity
>
> When signal states overlap, the q-machine density matrix has lower entropy than the classical case.
> This is the quantum advantage: $C_q < C_\mu$.